## Orange Distribution Logistics

A Sicilian company wants to sell oranges in various Italian regions. The oranges are grown and harvested in some Sicilian cities, each with different total production costs, leading to the following (selling price, maximum production): Catania (0.41 €/kg, 60 tons), Enna (0.36 €/kg, 30 tons), and Syracuse (0.31 €/kg, 50 tons). Oranges are moved and purchased exclusively in 150 kg boxes, so only quantities that are multiples of 150 kg may be bought.

The company has to organize the logistic to the following cities: Milan (the cost to reach Genoa by sea, plus 0.03 €/kg by land), Genoa (0.13 €/kg by sea), Rome (0.12 €/kg by freight train), Naples (0.10 €/kg by freight train). Moreover, if we pass from Genoa (to reach the Genoa or the Milan markets) we have to pay 3000 € of taxes every 200 kg of oranges that transit and if we ship from Enna to Naples we have to pay 1000 € due to some extra costs.

In the cities, associations of fruit sellers collectively request the following quantities of oranges from the company: Rome (40 tons), Milan (15 tons), Naples (35 tons), Genoa (20 tons). Oranges are sold at the following prices: Rome (2.1 €/kg), Milan (2.0 €/kg), Naples (1.7 €/kg), Genoa (1.9 €/kg).

The company wants to maximize profit, considering an extra penalty of 0.05 €/kg for lost sales and of 0.1 €/kg for any amount supplied above demand (regardless of the city).

Write an **MILP** to help the company (remember that 1ton = 1000kg).


**INDEXES**:
- $i$ : city of production
- $j$ : city of destination

**PARAMETERS**:
- $c_i$ : selling price from city i (/kg)
- $m_i$ : maximum production of city i (tons)
- $p_j$ : selling price at city j (/kg)
- $d_j$ : demand fo city j (tons)
- $t_{i,j}$ : transportation cost from i to j (/kg)

**VARIABLES**:
- $x_{i,j}$ : number of 150kg packages from i to j
- $y_j$ : number of 200kg block that pass through genova
- $z$ : {0,1} 1 if we ship from Enna to Naples
- $p_{l,j}$ : amount of kg of lost sales
- $p_{a,j}$ : amount of kg above the demand

$$ \max \sum_i \sum_j 150 \cdot x_{i,j} (p_j - c_i - t_{i,j}) - 3000y - 1000z - \sum_j (0.05 p_{l,j} + 0.10 p_{a,j}) $$

1. **Capacità massima di produzione** (convertita in kg):
$$ \sum_j 150 \cdot x_{i,j} \le 1000 \cdot m_i \quad \forall i $$

2. **Bilancio della domanda** (esatta compensazione tra invio, vendite perse e in eccesso):
$$ \sum_i 150 \cdot x_{i,j} + p_{l,j} - p_{a,j} = 1000 \cdot d_j \quad \forall j $$

3. **Calcolo della tassa per Genova** (blocchi da 200 kg per Genova e Milano):
$$ \sum_i 150 \cdot (x_{i, \text{Genoa}} + x_{i, \text{Milan}}) \le 200 \cdot y $$

4. **Attivazione della tassa extra Enna-Napoli** (tramite Big-M):
$$ 150 \cdot x_{\text{Enna}, \text{Naples}} \le M \cdot z $$
*(dove $M$ è un numero sufficientemente grande, ad esempio $1000 \cdot m_{\text{Enna}}$)*

5. **Natura delle variabili:**
$$ x_{i,j} \in \mathbb{Z}^+ \quad \forall i,j \quad \text{(pacchi interi non negativi)} $$
$$ y \in \mathbb{Z}^+ \quad \text{(numero di blocchi da 200kg, intero non negativo)} $$
$$ z \in \{0, 1\} \quad \text{(variabile binaria)} $$
$$ p_{l,j}, p_{a,j} \ge 0 \quad \forall j \quad \text{(variabili continue o intere non negative)} $$

In [1]:
import gurobipy as gp
from gurobipy import GRB

# =============================================================================
# MODEL: Orange Distribution Optimization (MILP)
# A Sicilian company wants to maximize profit from selling oranges across Italy.
# Oranges are shipped in 150 kg boxes and transit through Genoa (for Genoa and
# Milan markets) incurs a tax of €3000 per 200 kg block. Shipping from Enna
# to Naples incurs a fixed extra cost of €1000.
# =============================================================================

m = gp.Model("orange_distribution")

# =============================================================================
# PARAMETERS
# =============================================================================

# Cities of production (supply side)
producers = ["Catania", "Enna", "Syracuse"]

# Cities of destination (demand side)
destinations = ["Rome", "Naples", "Milan", "Genoa"]

# Production cost (selling price from city i) [€/kg]
c = {
    "Catania":  0.41,
    "Enna":     0.36,
    "Syracuse": 0.31
}

# Maximum production per city [tons] → will be converted to kg in constraints
m_max = {
    "Catania":  60,  # tons
    "Enna":     30,  # tons
    "Syracuse": 50   # tons
}

# Selling price at destination j [€/kg]
p = {
    "Rome":   2.1,
    "Naples": 1.7,
    "Milan":  2.0,
    "Genoa":  1.9
}

# Demand at destination j [tons] → will be converted to kg in constraints
d = {
    "Rome":   40,  # tons
    "Naples": 35,  # tons
    "Milan":  15,  # tons
    "Genoa":  20   # tons
}

# Transportation cost t_{i,j} [€/kg]
# Milan is reached via Genoa (sea €0.13 + land €0.03 = €0.16/kg)
# Rome and Naples are reached by freight train
# Genoa is reached by sea
t = {
    ("Catania",  "Rome"):   0.12,
    ("Catania",  "Naples"): 0.10,
    ("Catania",  "Milan"):  0.16,   # 0.13 sea (Genoa) + 0.03 land
    ("Catania",  "Genoa"):  0.13,
    ("Enna",     "Rome"):   0.12,
    ("Enna",     "Naples"): 0.10,
    ("Enna",     "Milan"):  0.16,
    ("Enna",     "Genoa"):  0.13,
    ("Syracuse", "Rome"):   0.12,
    ("Syracuse", "Naples"): 0.10,
    ("Syracuse", "Milan"):  0.16,
    ("Syracuse", "Genoa"):  0.13
}

# Big-M for the Enna→Naples activation constraint
# (safe upper bound: entire Enna production in kg)
BIG_M = m_max["Enna"] * 1000  # = 30,000 kg

# Penalty for lost sales [€/kg]
penalty_lost  = 0.05

# Penalty for supply above demand [€/kg]
penalty_above = 0.10

# Tax per 200-kg block transiting through Genoa (to reach Genoa or Milan)
tax_genoa_per_block = 3000   # [€ / 200 kg block]

# Fixed extra cost if we ship anything from Enna to Naples
extra_enna_naples = 1000     # [€]

# =============================================================================
# VARIABLES
# =============================================================================

# x[i, j]: number of 150-kg BOXES shipped from producer i to destination j
#           (non-negative integers)
x = {}
for i in producers:
    for j in destinations:
        x[i, j] = m.addVar(vtype=GRB.INTEGER, lb=0, name=f"x_{i}_{j}")

# y: number of 200-kg BLOCKS of oranges transiting through Genoa
#    (covers shipments to both Genoa and Milan, which also go via Genoa)
y = m.addVar(vtype=GRB.INTEGER, lb=0, name="y")

# z: binary variable — 1 if we ship at least 1 box from Enna to Naples, 0 otherwise
z = m.addVar(vtype=GRB.BINARY, name="z")

# p_l[j]: kg of LOST SALES at destination j (demand not met, continuous ≥ 0)
p_l = {}
for j in destinations:
    p_l[j] = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"p_l_{j}")

# p_a[j]: kg of EXCESS SUPPLY at destination j (above demand, continuous ≥ 0)
p_a = {}
for j in destinations:
    p_a[j] = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"p_a_{j}")

# =============================================================================
# OBJECTIVE FUNCTION
# Maximize:
#   revenue from sales
#   - Genoa transit tax (€3000 per 200-kg block)
#   - Enna→Naples fixed extra cost (€1000 if z=1)
#   - penalty for lost sales (€0.05/kg)
#   - penalty for excess supply (€0.10/kg)
# =============================================================================

revenue = gp.quicksum(
    150 * x[i, j] * (p[j] - c[i] - t[i, j])
    for i in producers
    for j in destinations
)

m.setObjective(
    revenue
    - tax_genoa_per_block * y
    - extra_enna_naples * z
    - gp.quicksum(penalty_lost  * p_l[j] for j in destinations)
    - gp.quicksum(penalty_above * p_a[j] for j in destinations),
    GRB.MAXIMIZE
)

# =============================================================================
# CONSTRAINTS
# =============================================================================

# 1. Maximum production capacity for each producer [kg]
#    sum_j (150 * x[i,j]) <= 1000 * m_max[i]  for all i
for i in producers:
    m.addConstr(
        gp.quicksum(150 * x[i, j] for j in destinations) <= 1000 * m_max[i],
        name=f"MaxProd_{i}"
    )

# 2. Demand balance at each destination [kg]
#    The total kg delivered, plus lost-sales kg, minus excess-supply kg = demand (in kg)
#    sum_i (150 * x[i,j]) + p_l[j] - p_a[j] = 1000 * d[j]  for all j
for j in destinations:
    m.addConstr(
        gp.quicksum(150 * x[i, j] for i in producers) + p_l[j] - p_a[j]
        == 1000 * d[j],
        name=f"DemandBalance_{j}"
    )

# 3. Genoa transit tax: every 200 kg going to Genoa OR Milan must be counted
#    in blocks y (because both routes go via Genoa by sea).
#    sum_i 150*(x[i,Genoa] + x[i,Milan]) <= 200 * y
m.addConstr(
    gp.quicksum(150 * (x[i, "Genoa"] + x[i, "Milan"]) for i in producers)
    <= 200 * y,
    name="GenoaTaxBlocks"
)

# 4. Big-M activation: z=1 iff at least 1 box from Enna to Naples is shipped
#    150 * x[Enna, Naples] <= BIG_M * z
m.addConstr(
    150 * x["Enna", "Naples"] <= BIG_M * z,
    name="EnnaToNaplesActivation"
)

# =============================================================================
# SOLVE
# =============================================================================

m.optimize()

# =============================================================================
# OUTPUT
# =============================================================================

print("-" * 50)
if m.status == GRB.OPTIMAL:
    print(f"Optimization status: OPTIMAL SOLUTION FOUND")
    print(f"Maximum Total Profit: €{m.objVal:,.2f}")
    print()

    print("--- Shipping Plan (boxes of 150 kg) ---")
    for i in producers:
        for j in destinations:
            boxes = x[i, j].x
            if boxes > 0.5:  # round-off tolerance
                kg = 150 * boxes
                print(f"  {i} -> {j}: {int(round(boxes))} boxes ({kg:.0f} kg)")

    print()
    print("--- Demand Balance per Destination ---")
    for j in destinations:
        total_kg = sum(150 * x[i, j].x for i in producers)
        demand_kg = 1000 * d[j]
        print(f"  {j}: shipped={total_kg:.0f} kg, demand={demand_kg:.0f} kg, "
              f"lost_sales={p_l[j].x:.0f} kg, excess={p_a[j].x:.0f} kg")

    print()
    print(f"--- Genoa Transit ---")
    total_genoa_kg = sum(150 * (x[i, "Genoa"].x + x[i, "Milan"].x) for i in producers)
    print(f"  Total kg via Genoa (Genoa + Milan): {total_genoa_kg:.0f} kg")
    print(f"  Number of 200-kg tax blocks (y): {int(round(y.x))}")
    print(f"  Genoa transit tax paid: €{tax_genoa_per_block * y.x:,.2f}")

    print()
    print(f"--- Enna -> Naples Extra Cost ---")
    enna_naples_kg = 150 * x["Enna", "Naples"].x
    print(f"  Enna->Naples: {enna_naples_kg:.0f} kg  (z={int(round(z.x))})")
    if z.x > 0.5:
        print(f"  Extra cost paid: €{extra_enna_naples:,.2f}")
    else:
        print(f"  No extra cost (nothing shipped from Enna to Naples).")

else:
    print("No optimal solution found.")
    print(f"Gurobi status code: {m.status}")


Set parameter Username
Set parameter LicenseID to value 2834064
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13620H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 9 rows, 22 columns and 41 nonzeros (Max)
Model fingerprint: 0xebe15a36
Model has 22 linear objective coefficients
Variable types: 8 continuous, 14 integer (1 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+04]
  Objective range  [5e-02, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+04, 6e+04]

Found heuristic solution: objective -5500.000000
Presolve time: 0.00s
Presolved: 9 rows, 22 columns, 41 nonzeros
Variable types: 8 continuous, 14 integer (1 binary)

Root relaxation: objective 2.127215e+05, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  